# Gas Classification — 모범답안2 (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 train.csv(센서·캡션)와 **열화상 이미지**(모범답안2가 사용)를 받아 `data/` 에 준비합니다. 이후 셀이 그대로 멀티모달 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 유형 변경 → GPU 권장.

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) — 모범답안2 는 train.csv + 열화상 이미지 사용 ===
import os, glob, zipfile
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    os.system('wget -q -O data/train.csv https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-competition-gas-classification/train.csv')
if not os.path.isdir('data/images'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('playground-joai-competition-2025', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
import os; print('데이터 준비:', 'train.csv' if os.path.exists('data/train.csv') else '실패', '| images', os.path.isdir('data/images'))

# JOAI 2025 — Gas Classification 모범답안 (multimodal late fusion)

> **Japan Olympiad in AI 2025 — Playground Competition**

Combine all three modalities by **late fusion** — average the class probabilities of three models:
1. **Sensors**: HistGradientBoosting on `MQ8`, `MQ5`
2. **Caption (text)**: TF-IDF + LogisticRegression
3. **Thermal image**: a fine-tuned ResNet18

Averaging the probabilities and taking the argmax beats any single modality. Held-out macro-F1.

In [ ]:
import pandas as pd, numpy as np, torch, torch.nn as nn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
device='cuda' if torch.cuda.is_available() else 'cpu'
df = pd.read_csv('data/train.csv').reset_index(drop=True)
is_val = df.index % 5 == 0
tr, va = df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)
CLASSES = sorted(df['Gas'].unique()); c2i={c:i for i,c in enumerate(CLASSES)}
print('classes', CLASSES, '| train', len(tr), 'val', len(va))

In [ ]:
# 1) sensors -> class probabilities
gbm = HistGradientBoostingClassifier(random_state=0).fit(tr[['MQ8','MQ5']], tr['Gas'])
p_sensor = pd.DataFrame(gbm.predict_proba(va[['MQ8','MQ5']]), columns=gbm.classes_)[CLASSES].to_numpy()
# 2) caption TF-IDF -> probabilities
vec = TfidfVectorizer(max_features=5000)
Xtr = vec.fit_transform(tr['Caption'].astype(str)); Xva = vec.transform(va['Caption'].astype(str))
lr = LogisticRegression(max_iter=2000).fit(Xtr, tr['Gas'])
p_text = pd.DataFrame(lr.predict_proba(Xva), columns=lr.classes_)[CLASSES].to_numpy()
print('sensor F1', round(f1_score(va['Gas'], np.array(CLASSES)[p_sensor.argmax(1)], average='macro'),4),
      '| text F1', round(f1_score(va['Gas'], np.array(CLASSES)[p_text.argmax(1)], average='macro'),4))

In [ ]:
import torchvision, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import f1_score

tf = T.Compose([T.Resize((96,96)), T.ToTensor(), T.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])])

class DS(Dataset):
    def __init__(s, d, lab=True): s.d=d.reset_index(drop=True); s.lab=lab
    def __len__(s): return len(s.d)
    def __getitem__(s,i):
        r=s.d.iloc[i]; img=tf(Image.open('data/images/'+r['image_path_uuid']).convert('RGB'))
        return (img, c2i[r['Gas']]) if s.lab else img

m = torchvision.models.resnet34(weights=torchvision.models.ResNet34_Weights.DEFAULT)
m.fc = nn.Linear(512, len(CLASSES)); m=m.to(device)
opt=torch.optim.AdamW(m.parameters(),3e-4); crit=nn.CrossEntropyLoss()
sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=5)

dl = DataLoader(DS(tr), 64, shuffle=True, num_workers=4)
val_dl = DataLoader(DS(va, lab=True), 128, shuffle=False, num_workers=2)

best_f1 = 0.0
best_model_path = 'best_resnet34.pth'

for ep in range(10):
    m.train()
    for x, y in dl: 
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        crit(m(x), y).backward()
        opt.step()
    sch.step()
    
    m.eval()
    val_preds = []
    val_targets = []
    
    with torch.no_grad():
        for x, y in val_dl:
            x = x.to(device)
            preds = torch.argmax(m(x), dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(y.numpy())
            
    current_f1 = f1_score(val_targets, val_preds, average='macro')
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        torch.save(m.state_dict(), best_model_path)
        print(f'epoch {ep+1} done | F1: {current_f1:.4f} (Best Saved)')
    else:
        print(f'epoch {ep+1} done | F1: {current_f1:.4f}')

m.load_state_dict(torch.load(best_model_path))
m.eval()

probs = []
with torch.no_grad():
    for x in DataLoader(DS(va, lab=False), 128, num_workers=2):
        probs.append(torch.softmax(m(x.to(device)), 1).cpu().numpy())
        
p_img = np.concatenate(probs)
print('image F1', round(f1_score(va['Gas'], np.array(CLASSES)[p_img.argmax(1)], average='macro'), 4))

In [ ]:
# late fusion: average the three probability matrices
p = (p_sensor + p_text + p_img) / 3
pred = np.array(CLASSES)[p.argmax(1)]
print('FUSION held-out macro-F1:', round(f1_score(va['Gas'], pred, average='macro'), 4))
pd.DataFrame({'index': range(len(va)), 'Gas': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(va))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)